<a href="https://colab.research.google.com/github/Hamerson-jhoel/Procesamiento-de-Lengaje-Natural/blob/main/PARCIAL_PLN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Universidad Nacional de Colombia sede Manizales
#Procesamiento del Lenguaje Natural
#Hamerson Jhoel Piarpuezan Piarpuezan
- CC. 1004531735

#Ejercicio 1: Manipulación y Regex de Diálogos
- Contexto: Tienes la siguiente cadena de texto que representa un log de un sistema de atención:


      log = "User: [JuanP] | Time: 14:30:05 | Msg: Necesito ayuda con mi suscripción #12345"

- Instrucción: Escribe un código en Python que utilice un único patrón de Expresiones Regulares con grupos con nombre (Named Groups) para extraer el nombre del usuario, la hora y el número de ticket (el valor numérico después del #). Almacena el resultado en un diccionario.

- Pregunta para el examen: > ¿Qué sucede con tu expresión regular si el nombre del usuario contiene caracteres especiales (ej: [Juan_P.23]) y cómo la ajustarías para que sea robusta sin usar grupos genéricos como .*??

In [1]:
import re
# Importamos el módulo 're', que permite trabajar con expresiones regulares

log = "User: [JuanP] | Time: 14:30:05 | Msg: Necesito ayuda con mi suscripción #12345"
# Definimos el patrón regex usando una cadena RAW multilínea.
# Esto permite escribir el patrón organizado y usar re.VERBOSE
pattern = r"""
User:\s\[(?P<user>[A-Za-z0-9]+)\]
\s\|\s
Time:\s(?P<time>\d{2}:\d{2}:\d{2})
\s\|\s
Msg:.*\#(?P<ticket>\d+)
"""

# re.search busca el patrón dentro del texto 'log'.
# re.VERBOSE permite usar espacios y comentarios dentro del regex.
match = re.search(pattern, log, re.VERBOSE)

# Verificamos si el patrón encontró coincidencias.
if match:
    # groupdict() crea un diccionario con los grupos con nombre.
    # Las claves serán: user, time y ticket.
    result = match.groupdict()
    # Imprimimos el diccionario con la información extraída.
    print(result)

{'user': 'JuanP', 'time': '14:30:05', 'ticket': '12345'}


# Ejercicio 1: Parte Teórica

### **Pregunta:**
¿Qué sucede si el nombre de usuario contiene caracteres especiales (por ejemplo: `Juan_P.23`)?

---

### **Análisis del Problema**

Si utilizamos el patrón original `[A-Za-z0-9]+`, el sistema solo permitirá caracteres alfanuméricos simples. Al procesar una cadena como `Juan_P.23`, el motor de expresiones regulares encontrará caracteres no permitidos:

* **Patrón actual:** `[A-Za-z0-9]+` → Solo letras y números.
* **Entrada:** `Juan_P.23` → Contiene guion bajo (`_`) y punto (`.`).
* **Resultado:**  **Error de validación / Fallo.**

### ** Solución Robusta**

Para solucionar esto sin comprometer la seguridad (evitando el uso de comodines genéricos como `.*`), debemos expandir nuestra **clase de caracteres**.

**Modificación sugerida:**
> De: `[A-Za-z0-9]+`  
> A:  `[A-Za-z0-9_.]+`

Con este cambio, el patrón ahora acepta explícitamente:
*  Letras (mayúsculas y minúsculas)
*  Números
*  Guiones bajos (`_`)
*  Puntos (`.`)

---

### **Respuesta Teórica (Para Examen)**

> *"Si el nombre contiene caracteres especiales como '_' o '.', la expresión regular original fallaría porque su conjunto de caracteres está limitado a alfanuméricos. Para hacerla robusta, se debe modificar el patrón a **`[A-Za-z0-9_.]+`**. Esto permite la inclusión específica de guiones bajos y puntos, manteniendo el control sobre la cadena sin recurrir a expresiones demasiado permisivas como `.*`."*

In [2]:
import re

log = "User: [Juan_P.] | Time: 14:30:05 | Msg: Necesito ayuda con mi suscripción #12345"

pattern = r"""
User:\s\[(?P<user>[A-Za-z0-9_.]+)\]
\s\|\s
Time:\s(?P<time>\d{2}:\d{2}:\d{2})
\s\|\s
Msg:[^#]*\#(?P<ticket>\d+)
"""

match = re.search(pattern, log, re.VERBOSE)

if match:
    result = match.groupdict()
    print(result)

{'user': 'Juan_P.', 'time': '14:30:05', 'ticket': '12345'}



1. **Comportamiento Greedy (Codicioso): `.*\#`**
   * **Descripción:** Consume cualquier carácter hasta el **último** símbolo `#` que encuentre en la línea.
   * **Problema:** Si existen múltiples `#`, provocará capturas incorrectas al "pasarse" de los primeros separadores (sobrecaptura).

2. **Comportamiento Robusto (Preciso): `[^#]*\#`**
   * **Descripción:** Consume caracteres solo hasta encontrar el **primer** `#`.
   * **Ventaja:** Al usar una clase de caracteres negada (`[^#]`), se asegura de detenerse inmediatamente en el primer delimitador, evitando errores y haciendo la expresión mucho más precisa.


> **Conclusión :** > El uso de `[^#]*\#` es preferible para evitar sobrecapturas, garantizando que la búsqueda sea robusta y se detenga exactamente en el punto deseado.

#Ejercicio 2: Normalización y Limpieza con SpaCy
Contexto: Un estudiante propone la siguiente función para limpiar una lista de tokens:

    def limpiar(tokens):
        return [t.lower() for t in tokens if t not in stop_words]

Instrucción: Mejora esta función integrando SpaCy. El nuevo código debe recibir un texto (no una lista), convertirlo a minúsculas, eliminar signos de puntuación, eliminar stopwords y devolver únicamente los lemas de los Sustantivos (NOUN) y Verbos (VERB).

-Pregunta para el examen:

Si el modelo

    en_core_web_sm
  
  de SpaCy clasifica erróneamente una palabra (por ejemplo, trata un nombre propio como un verbo), ¿qué componente del pipeline de SpaCy podrías modificar o qué técnica usarías para forzar la categoría correcta?

In [3]:
import spacy

nlp = spacy.load("en_core_web_sm")

def limpiar_texto(texto):

    # convertir a minúsculas
    texto = texto.lower()

    # procesar texto
    doc = nlp(texto)

    resultado = []

    for token in doc:

        # eliminar puntuación
        if token.is_punct:
            continue

        # eliminar stopwords
        if token.is_stop:
            continue

        # solo NOUN y VERB
        if token.pos_ not in ["NOUN", "VERB"]:
            continue

        # agregar lema
        resultado.append(token.lemma_)

    return resultado

In [4]:
texto = "The students were running quickly to the university??!!!."

resultado = limpiar_texto(texto)

print(resultado)

['student', 'run', 'university']


#Implementacion parte interactiva

In [5]:
import spacy

# Cargar modelo una sola vez
nlp = spacy.load("en_core_web_sm")

# Esta función recibe un texto y devuelve una lista de tokens limpios.
def limpiar_spacy(texto):
    # Convertimos todo el texto a minúscula
    texto = texto.lower()
    # Procesamos el texto con SpaCy.
    # Esto genera un objeto "doc" que contiene:
    # - tokens
    # - etiquetas gramaticales
    # - lemas
    # - información lingüística
    doc = nlp(texto)
    # Creamos una lista vacía donde guardaremos
    # los tokens limpios y lematizados.
    resultado = []


   # Recorremos cada token del documento
    for token in doc:

        # eliminar puntuación
        if token.is_punct:
            continue

        # eliminar stopwords
        # "the", "is", "and", "of"
        if token.is_stop:
            continue

        # eliminar números
        # Detecta tanto:
        # "123"
        # "one"
        if token.like_num:
            continue

        # Mantiene solo Sustantivos y Verbos
        if token.pos_ not in ["NOUN", "VERB"]:
            continue

        # -------- LEMATIZACIÓN --------
        # token.lemma_ devuelve la forma base de la palabra.
        # Ejemplo:
        # "running" → "run"
        # "dogs" → "dog"
        resultado.append(token.lemma_)

    return resultado

    # -------- PARTE INTERACTIVA --------
# Permite al usuario escribir textos continuamente.

while True:

    # Pedimos al usuario que escriba un texto.
    # También puede escribir "salir" para terminar.
    texto = input("Escribe un texto (o 'salir'): ")

    # Si el usuario escribe "salir",
    # el programa termina.
    if texto.lower() == "salir":
        print("Programa terminado.")
        break

    # Llamamos a la función limpiar_spacy
    # y procesamos el texto ingresado.
    resultado = limpiar_spacy(texto)

    # Mostramos el resultado limpio
    print("Resultado:", resultado)

    # Imprime una línea en blanco para separar resultados
    print()

Escribe un texto (o 'salir'): The students were running quickly to the university
Resultado: ['student', 'run', 'university']

Escribe un texto (o 'salir'): salir
Programa terminado.


# Que convierta a minusculas despues de los filtros para detectar PRONOMBRES

# Pregunta:
- Si el modelo en_core_web_sm clasifica erróneamente una palabra (por ejemplo, trata un nombre propio como un verbo), ¿qué componente del pipeline podrías modificar o qué técnica usarías?


- El POS Tagger es parte del pipeline de SpaCy y determina si una palabra es sustantivo, verbo, adjetivo, etc. Si falla, se puede reentrenar el modelo con ejemplos etiquetados correctamente o usar reglas personalizadas para mejorar la clasificación.

- Se puede corregir usando reglas manuales (rule-based)
- Se puede mejorar reentrenando el modelo (fine-tuning)
- Se puede aplicar post-procesamiento de etiquetas
- Si el modelo clasifica incorrectamente una palabra, se puede corregir usando reglas manuales, aplicar un Matcher para detectar patrones específicos o realizar fine-tuning del modelo entrenándolo con ejemplos correctamente etiquetados.

#Ejercicio 3: Ingeniería de Características Manuales
Contexto: Estás trabajando en un detector de SPAM. Tienes un DataFrame con una columna text.

Instrucción: Implementa una función que cree dos nuevas columnas:

    url_count: Cuenta cuántas veces aparece "http" o "https" en el mensaje.

    upper_ratio: La proporción de letras mayúsculas respecto al total de letras del mensaje.
Pregunta para el examen:

Si el modelo de Regresión Logística entrenado con estas características tiene un Accuracy muy alto pero un Recall muy bajo para la clase SPAM, ¿qué te indica esto sobre la utilidad de tus características manuales?

- url_count

Cuenta cuántas veces aparece:

    http
    https
- upper_ratio

Es:

    (# letras MAYÚSCULAS) / (# total letras)

In [6]:
# Importamos pandas para manejar DataFrames
import pandas as pd

# Importamos re para usar expresiones regulares
import re


# Definimos la función que crea nuevas features
def crear_features(df):

    # -------- FEATURE 1: url_count --------
    # Creamos la columna "url_count"
    # Ahora usamos regex para contar correctamente
    # "http" o "https" sin duplicaciones.

    df["url_count"] = df["text"].apply(

        # re.findall busca coincidencias del patrón
        # r"https?" significa:
        # http  → coincide
        # https → coincide (la "s" es opcional)
        # len(...) cuenta cuántas coincidencias se encontraron
        lambda x: len(re.findall(r"https?", x))
    )


    # -------- FEATURE 2: upper_ratio --------
    # Función para calcular la proporción de mayúsculas
    def upper_ratio(texto):

        # Extraemos solo letras del texto
        letras = [c for c in texto if c.isalpha()]

        # Evitamos división por cero
        if len(letras) == 0:
            return 0

        # Extraemos solo letras mayúsculas
        mayusculas = [c for c in letras if c.isupper()]

        # Calculamos proporción
        return len(mayusculas) / len(letras)


    # Aplicamos la función a cada texto
    df["upper_ratio"] = df["text"].apply(upper_ratio)


    # Retornamos el DataFrame actualizado
    return df


# -------- DATOS DE PRUEBA --------

# Creamos un DataFrame con mensajes ejemplo
df = pd.DataFrame({
    "text": [
        "WINNER! Click http://spam.com NOW!!!",
        "Hello mom, I will be home soon.",
        "Visit https://offer.com TODAY"
    ]
})


# Aplicamos la función
df = crear_features(df)


# Mostramos resultado
print(df)

                                   text  url_count  upper_ratio
0  WINNER! Click http://spam.com NOW!!!          1     0.400000
1       Hello mom, I will be home soon.          0     0.086957
2         Visit https://offer.com TODAY          1     0.260870


### Accuracy

**Accuracy (Exactitud)** =
De todos los mensajes, cuántos clasificó bien el modelo.

**Fórmula conceptual:**
$$\text{Accuracy} = \frac{\text{Predicciones correctas}}{\text{Total de predicciones}}$$

---

### Recall

**Recall (Sensibilidad)** mide:
De todos los SPAM reales, cuántos detectó.

**Fórmula conceptual:**
$$\text{Recall} = \frac{\text{SPAM detectados}}{\text{SPAM reales}}$$

- Si el modelo de Regresión Logística entrenado con estas características tiene un Accuracy muy alto pero un Recall muy bajo para la clase SPAM, ¿qué te indica esto sobre la utilidad de tus características manuales?

# Respuesta

**La interpretación correcta es:**
**Tus características no están ayudando a detectar SPAM**

Es decir:
* El modelo clasifica bien los HAM
* Pero no identifica bien los SPAM
* Entonces las features no capturan bien el patrón del spam

---

###  Ejemplo aplicado a los features
Tus features son:
1. `url_count`
2. `upper_ratio`

**Supón esto:**
Mensaje SPAM: *"Win money now!!!"*

No tiene:
* http
* muchas mayúsculas

**Entonces:**
* `url_count` = 0
* `upper_ratio` = 0.1

 **Parece un mensaje normal**
,  **El modelo lo clasifica como HAM**

**Resultado:**
 SPAM no detectado  ➡ **Recall baja**

---

###  Lo que realmente significa
Si pasa:
✔ **Accuracy alto**
❌ **Recall bajo**

**Significa:**
-  Tus características no son buenas para detectar SPAM. **Necesitas mejores features.**

---

###  Ejemplos de features mejores
Por ejemplo:
* `num_exclamations`
* `num_numbers`
* `contains_money_word`
* `message_length`
* `num_special_characters`

**Ejemplo:** *"WIN $1000 NOW!!!"*
Podría dar:
* `num_exclamations` = 3
* `contains_money_word` = 1
* `upper_ratio` = 0.8

 **Eso sí ayuda a detectar spam.**

---

### Respuesta

**Pregunta:** ¿Qué indica un Accuracy alto pero Recall bajo para SPAM?

**Respuesta recomendada:**
> Indica que el modelo clasifica correctamente la mayoría de los mensajes, pero falla en detectar los SPAM. Esto sugiere que las características manuales utilizadas no capturan adecuadamente los patrones del spam, por lo que deben mejorarse o agregarse nuevas características.


# Ejercicio 4: Representación BoW y TF-IDF
Contexto: Tienes un corpus de 2 frases:

    "El gato duerme"
    "El perro corre"
Instrucción: Calcula manualmente (o describe el proceso) la matriz resultante de Bag of Words (BoW). Luego, explica cómo cambiaría el valor de la palabra "El" si aplicáramos TF-IDF.

Pregunta para el examen:

En un corpus muy grande, la matriz BoW suele ser "dispersa" (sparse). ¿Qué significa esto en términos de memoria y qué problemas puede causar al entrenar un modelo de Machine Learning?

# SOLUCION— Bag of Words (BoW) y TF-IDF

### 🔹 Corpus dado
Tienes 2 frases:
1. "El gato duerme"
2. "El perro corre"

Te piden:
* Construir matriz BoW manual
* Explicar qué pasa con "El" usando TF-IDF
* Explicar qué significa que BoW sea sparse

---

### ✅ Parte 1 — Construcción manual de Bag of Words (BoW)

**Paso 1: Crear vocabulario**
Extraemos todas las palabras únicas:
* El
* gato
* duerme
* perro
* corre

**Vocabulario final:**
`["El", "gato", "duerme", "perro", "corre"]`

**Paso 2: Contar palabras por frase**

**Frase 1:** "El gato duerme"
* El → 1
* gato → 1
* duerme → 1
* perro → 0
* corre → 0
* **Vector:** `[1, 1, 1, 0, 0]`

**Frase 2:** "El perro corre"
* El → 1
* gato → 0
* duerme → 0
* perro → 1
* corre → 1
* **Vector:** `[1, 0, 0, 1, 1]`

**✅ Matriz BoW final**

| | El | gato | duerme | perro | corre |
|---|---:|---:|---:|---:|---:|
| **Frase 1 →** | 1 | 1 | 1 | 0 | 0 |
| **Frase 2 →** | 1 | 0 | 0 | 1 | 1 |

Esta es la respuesta correcta del BoW.

---

###  Parte 2 — ¿Qué pasa con "El" usando TF-IDF?

Ahora viene lo conceptual.
TF-IDF penaliza palabras que aparecen en muchos documentos.
La palabra: **"El"** aparece en: **Frase 1 ✔ y Frase 2 ✔**

Entonces:
* `document_frequency = 2`
* Eso significa: **IDF será bajo**
* Y por tanto: **TF-IDF("El") ↓**

**En TF-IDF, la palabra "El" tendría un valor menor que en BoW, porque aparece en todos los documentos del corpus. TF-IDF reduce el peso de palabras muy frecuentes, ya que aportan poca información para diferenciar los textos.**
> La palabra "El" aparece en todos los documentos, por lo que su frecuencia inversa de documento (IDF) será baja. Como resultado, su valor TF-IDF disminuirá, ya que se considera una palabra poco informativa para diferenciar documentos.

**Idea clave que debes recordar**
TF-IDF hace esto:
* palabra común → peso bajo
* palabra rara → peso alto

**Ejemplo:**
* El → bajo peso
* gato → peso mayor
* perro → peso mayor

---

### Parte 3 — Pregunta sobre matrices "sparse"

**Te preguntan:** En un corpus muy grande, la matriz BoW suele ser "dispersa" (sparse). ¿Qué significa esto y qué problemas causa?

**Respuesta  para examen**
> Una matriz sparse significa que la mayoría de sus valores son cero, ya que cada documento contiene solo una pequeña parte del vocabulario total. Esto aumenta el uso de memoria y puede hacer que el entrenamiento de modelos sea más lento y computacionalmente costoso.

**Explicación**
Imagina:
* Vocabulario = 10,000 palabras
* Documento usa = 20 palabras
* Entonces: `vector = [0,0,0,0,0,1,0,0,0,0,0,...]`
* La mayoría son: **0**
* Eso es: **SPARSE**

**Problemas que causa**
Muy importante para examen:
1. Alto consumo de memoria
2. Cálculos lentos
3. Modelos más difíciles de entrenar



### **Problema 1 — Uso de memoria**

Si tienes:
* 1 millón de documentos
* 50,000 palabras

Entonces:
$1,000,000 \times 50,000$

Eso es:
**50 mil millones de valores**

Aunque:
La mayoría son 0

Pero igual ocupan memoria. Eso hace que:
El modelo consuma mucha RAM
y Sea lento

---

### **Problema 2 — Entrenamiento lento**

Muchos ceros implican:
* Operaciones innecesarias
* Cálculos lentos
* Modelos más pesados

**Ejemplo:**
Entrenar *Logistic Regression* puede tardar mucho.

---

### **Problema 3 — Modelos mas dificiles de entrenar (sobre ajuste)**

Muchísimas dimensiones:
**50,000 features**

Eso provoca:
* Sobreajuste (*overfitting*)
* Mala generalización

---

### **Resumen**

Una matriz dispersa (**sparse**) significa que la mayoría de sus valores son cero. Esto ocurre porque cada documento contiene solo unas pocas palabras del vocabulario total. En términos de memoria, esto puede consumir muchos recursos, ya que se almacenan muchos ceros innecesarios. Además, puede hacer que el entrenamiento de modelos sea más lento y aumentar el riesgo de sobreajuste debido a la alta dimensionalidad.

# **Ejercicio 5: Embeddings y Modelos Contextuales**
Contexto: Estás comparando Word2Vec (GloVe) frente a BERT.

  Instrucción: Explica mediante un ejemplo de código (pseudocódigo o PyTorch) cómo obtendrías el vector que representa a una oración completa usando un modelo GloVe (estático) vs un modelo BERT.

- Pregunta para el examen:

Imagina la palabra "Banco". ¿Qué diferencia habría en el vector (embedding) generado por Word2Vec y el generado por BERT en las frases: "Me senté en el banco" y "Cobré el cheque en el banco"?

# **Embedding de oración usando GloVe**

### **Primero entendamos:**
**GloVe genera vectores por palabra**, pero no por oración.

Entonces, para obtener el vector de una oración:
**Promediamos los vectores de sus palabras.**

---

### **Ejemplo conceptual**

**Frase:** > "El gato duerme"

**Supón los siguientes vectores:**
* **El** $\rightarrow [0.1, 0.2]$
* **gato** $\rightarrow [0.5, 0.3]$
* **duerme** $\rightarrow [0.2, 0.8]$

**Entonces, el Vector de la oración es el promedio:**

$$\text{Vector Final} = \frac{[0.1, 0.2] + [0.5, 0.3] + [0.2, 0.8]}{3}$$

**Resultado del promedio:**
$[0.26, 0.43]$

# **Embedding usando BERT**

Ahora cambia todo.
**BERT genera embeddings dependientes del contexto.**

Además:
Tiene un token especial: **`[CLS]`**
Ese token representa: **toda la oración.**

**Idea clave:**
BERT suele usar el **embedding del token `[CLS]`** para representar la frase, no un promedio simple.

---

### Comparativa: Word2Vec / GloVe vs. BERT

#### **Word2Vec / GloVe (Estáticos)**
Generan un solo vector por palabra.
* **banco** → mismo vector siempre.
* No importa el contexto.

**Resultado con Word2Vec:**
1. *"Me senté en el **banco**"*
2. *"Cobré el cheque en el **banco**"*

La palabra **banco** tendrá **exactamente el mismo embedding** en ambas frases. Esto es un problema porque **no distingue significados**.

#### **BERT (Contextuales)**
BERT sí usa contexto. Los embeddings serán distintos según el sentido:

1. *"Me senté en el **banco**"* → **banco** (asiento)
2. *"Cobré el cheque en el **banco**"* → **banco** (institución financiera)

---

### Ejemplo conceptual visual

**Word2Vec:**
* `banco` → `[0.45, 0.21, 0.89]`
* `banco` → `[0.45, 0.21, 0.89]`
*(Siempre igual)*

**BERT:**
* `banco (sentarse)` → `[0.12, 0.77, 0.03]`
* `banco (dinero)` → `[0.91, 0.02, 0.55]`
*(Vectores dinámicos)*

---

### Respuesta recomendada para examen

**Pregunta:** ¿Qué diferencia habría en los embeddings de la palabra "Banco"?

**Respuesta:**
> En Word2Vec o GloVe, la palabra "Banco" tendría el mismo vector en ambas frases, ya que estos modelos generan embeddings estáticos que no dependen del contexto. En cambio, en BERT, el embedding sería diferente en cada frase, ya que BERT genera representaciones contextuales y ajusta el vector de la palabra según su significado dentro de la oración.

- En modelos como Word2Vec o GloVe, la palabra "banco" tendría el mismo vector en ambas frases, ya que estos modelos generan embeddings estáticos. En cambio, BERT genera embeddings contextuales, por lo que la palabra "banco" tendría vectores diferentes en "Me senté en el banco" y "Cobré el cheque en el banco", ya que el contexto cambia su significado.
- BERT genera embeddings contextuales usando self-attention, donde cada palabra ajusta su vector según las otras palabras en la misma oración, y el token [CLS] se utiliza como representación de la oración completa.

#Pseudocodigo Glove

    INICIO

    Cargar embeddings GloVe

    FUNCION obtener_vector_glove(oracion):

    Convertir oracion a minúsculas

    Separar la oracion en palabras

    Crear lista vacía llamada vectores

    PARA cada palabra EN la oracion:

        SI la palabra existe en el vocabulario GloVe:

            Obtener vector de la palabra

            Agregar vector a la lista vectores

    SI la lista vectores no está vacía:

        Calcular el promedio de todos los vectores

        vector_oracion ← promedio

    RETORNAR vector_oracion

    FIN FUNCION

    FIN

#codigo

In [7]:
# Instalamos gensim (solo se hace una vez)
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 64.5 MB/s eta 0:00:00


In [8]:
# =========================
# GLOVE: VECTOR DE ORACIÓN
# =========================

# Importamos numpy para trabajar con vectores numéricos
import numpy as np

# Importamos el descargador de modelos preentrenados
# desde la librería gensim
import gensim.downloader as api


# Cargamos un modelo GloVe preentrenado.
# "glove-wiki-gigaword-50" significa:
# - Entrenado con Wikipedia
# - Cada palabra tiene vector de 50 dimensiones
print("Cargando modelo GloVe...")
glove_model = api.load("glove-wiki-gigaword-50")


# Guardamos la dimensión del vector.
# En este modelo será 50.
embedding_dim = glove_model.vector_size

print("Dimensión del vector:", embedding_dim)


# Definimos una función que convierte
# una oración en un vector promedio.
def document_vector(doc):

    # Convertimos texto a minúsculas
    # y lo dividimos en palabras.
    tokens = doc.lower().split()

    # Filtramos solo las palabras
    # que existen en el vocabulario GloVe.
    tokens_validos = [
        word for word in tokens
        if word in glove_model
    ]

    # Si ninguna palabra existe en el vocabulario,
    # devolvemos un vector de ceros.
    if not tokens_validos:
        return np.zeros(embedding_dim)

    # Obtenemos los vectores
    # de todas las palabras válidas.
    vectores = glove_model[tokens_validos]

    # Calculamos el promedio
    # de todos los vectores.
    vector_oracion = np.mean(
        vectores,
        axis=0
    )

    # Retornamos el vector final.
    return vector_oracion


# -------- FRASES DE PRUEBA --------

# Definimos tus dos frases.
frases = [

    # Contexto tipo asiento
    "Me sente en el banco",

    # Contexto financiero
    "Retire dinero en el banco"
]


# Procesamos cada frase
for frase in frases:

    # Obtenemos el vector de la oración
    vec = document_vector(frase)

    # Mostramos la frase
    print("\nFrase:", frase)

    # Mostramos la dimensión del vector
    print("Dimensión vector:", len(vec))

    # Mostramos solo los primeros 10 valores
    # (porque mostrar los 50 sería muy largo)
    print("Primeros 10 valores del vector:")

    print(vec[:10])

    print("\n" + "-"*50)

Cargando modelo GloVe...
[==================================================] 100.0% 66.0/66.0MB downloaded
Dimensión del vector: 50

Frase: Me sente en el banco
Dimensión vector: 50
Primeros 10 valores del vector:
[ 0.13864641  0.237878   -0.287826    0.506494   -0.26805598 -1.33585
  0.041822    0.15391262 -0.380216    0.76993793]

--------------------------------------------------

Frase: Retire dinero en el banco
Dimensión vector: 50
Primeros 10 valores del vector:
[ 0.05541899  0.2905218  -0.143236    0.72390604 -0.401678   -1.466144
 -0.00691801  0.30915803 -0.43799058  0.707422  ]

--------------------------------------------------


In [9]:
vec_banco_1 = glove_model["banco"]
vec_banco_2 = glove_model["banco"]

print(np.allclose(vec_banco_1, vec_banco_2))

True


#Pseudocodigo BERT

    INICIO

    Cargar tokenizer de BERT

    Cargar modelo BERT

    FUNCION obtener_vector_bert(oracion):

    Tokenizar la oracion

    Agregar token [CLS] al inicio

    Agregar token [SEP] al final

    Convertir tokens a índices numéricos

    Enviar los índices al modelo BERT

    Obtener los embeddings de salida

    Extraer el embedding del token [CLS]

    vector_oracion ← embedding de [CLS]

    RETORNAR vector_oracion

    FIN FUNCION

    FIN

In [10]:
# Instalar librerías necesarias para BERT
!pip install transformers torch tqdm numpy -q

1. Descarga archivos del modelo
2. Descarga el vocabulario
3. Carga los pesos matemáticos
4. Prepara el modelo en memoria

In [11]:
# =========================
# BERT: VECTOR DE ORACIÓN
# =========================

# Importamos numpy para manejar vectores numéricos
import numpy as np

# Importamos torch (PyTorch), que permite ejecutar modelos BERT
import torch

# Importamos tokenizer y modelo BERT desde transformers
from transformers import BertTokenizer, BertModel


# -------- CONFIGURACIÓN DE DISPOSITIVO --------

# Detecta si hay GPU disponible.
# Si no hay GPU, usa CPU.
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Dispositivo usado:", device)


# -------- CARGA DEL MODELO --------

# Cargamos el tokenizador BERT.
# El tokenizador convierte texto en tokens que BERT entiende.
tokenizer = BertTokenizer.from_pretrained(
    "bert-base-uncased"
)

# Cargamos el modelo BERT.
# "bert-base-uncased" significa:
# - Modelo base
# - Usa minúsculas
# - Vector de tamaño 768
model = BertModel.from_pretrained(
    "bert-base-uncased"
)

# Movemos el modelo al dispositivo (CPU o GPU)
model = model.to(device)

# Ponemos el modelo en modo evaluación.
# Esto desactiva entrenamiento y ahorra memoria.
model.eval()


# -------- FUNCIÓN PARA VECTOR DE ORACIÓN --------

def bert_sentence_vector(text):

    """
    Convierte una oración en un vector usando
    el token especial [CLS].
    """

    # Tokenizamos el texto.
    # Esto convierte texto en números (IDs).
    inputs = tokenizer(
        text,
        return_tensors="pt",  # Devuelve tensores PyTorch
        padding=True,         # Rellena si hace falta
        truncation=True       # Corta si es muy largo
    )

    # Movemos datos al dispositivo
    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    # Desactivamos cálculo de gradientes
    # (no estamos entrenando).
    with torch.no_grad():

        # Pasamos el texto por BERT
        outputs = model(**inputs)

        # Extraemos la última capa oculta.
        # Forma:
        # [batch, longitud, 768]
        last_hidden = outputs.last_hidden_state

        # Extraemos el token [CLS]
        # Índice 0 siempre es [CLS].
        cls_vector = last_hidden[:, 0, :]

        # Convertimos a numpy
        cls_vector = cls_vector.cpu().numpy()

    # Retornamos el vector final
    return cls_vector[0]


# -------- FRASES DE PRUEBA --------

# Usamos tus dos frases
frases = [

    # Contexto de asiento
    "Me sente en el banco",

    # Contexto financiero
    "Retire dinero en el banco"
]


# -------- GENERACIÓN DE VECTORES --------

for frase in frases:

    # Obtener vector de la oración
    vec = bert_sentence_vector(frase)

    # Mostrar la frase original
    print("\nFrase:", frase)

    # Mostrar dimensión del vector
    print("Dimensión vector:", len(vec))

    # Mostrar primeros valores del vector
    print("Primeros 10 valores del vector [CLS]:")

    print(vec[:10])

    print("\n" + "-"*60)

Dispositivo usado: cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Frase: Me sente en el banco
Dimensión vector: 768
Primeros 10 valores del vector [CLS]:
[-0.48360097 -0.14510573 -0.1606844   0.04358227 -0.2704535  -0.01129399
  0.6521445   0.7888595   0.3968978  -0.27500385]

------------------------------------------------------------

Frase: Retire dinero en el banco
Dimensión vector: 768
Primeros 10 valores del vector [CLS]:
[-0.2986854  -0.29957238  0.00374061  0.34424606 -0.40453637 -0.05504571
  0.4845518   1.04185     0.13777103 -0.29612526]

------------------------------------------------------------


In [12]:
# Obtener tokens y embeddings
inputs1 = tokenizer("Me sente en el banco", return_tensors="pt")
inputs2 = tokenizer("Retire dinero en el banco", return_tensors="pt")

with torch.no_grad():

    out1 = model(**inputs1).last_hidden_state
    out2 = model(**inputs2).last_hidden_state

# Buscar posición de "banco"
tokens1 = tokenizer.convert_ids_to_tokens(inputs1["input_ids"][0])
tokens2 = tokenizer.convert_ids_to_tokens(inputs2["input_ids"][0])

idx1 = tokens1.index("banco")
idx2 = tokens2.index("banco")

# Extraer vectores de "banco"
vec_banco_1 = out1[0, idx1, :].numpy()
vec_banco_2 = out2[0, idx2, :].numpy()

# Comparar vectores
print(np.allclose(vec_banco_1, vec_banco_2))

False
